In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
import re
import html
import matplotlib.pyplot as plt

# Find project root
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "backend").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "backend").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError("TrustLens project folder not found.")

# Dataset paths
DATA_DIR = PROJECT_ROOT / "backend" / "data"

INPUT_PATH = DATA_DIR / "WELFake_cleaned.csv"
OUTPUT_PATH = DATA_DIR / "WELFake_preprocessed.csv"

print("Project root:", PROJECT_ROOT)
print("Input dataset:", INPUT_PATH)
print("Input exists:", INPUT_PATH.exists())

Project root: c:\Projects\fake_news_detection_extension
Input dataset: c:\Projects\fake_news_detection_extension\backend\data\WELFake_cleaned.csv
Input exists: True


In [3]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        "WELFake_cleaned.csv not found. "
        "Save the cleaned dataset from your cleaning notebook first."
    )

df = pd.read_csv(INPUT_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

Dataset loaded successfully!
Dataset shape: (63676, 3)
Columns: ['title', 'text', 'label']


,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,NaN,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last n...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [4]:
for i in range(3):

    print(f"\nARTICLE {i + 1}")
    print("=" * 70)

    print("TITLE:")
    print(df.loc[i, "title"])

    print("\nARTICLE TEXT:")
    print(df.loc[i, "text"][:500])

    print("\nLABEL:")
    print(df.loc[i, "label"])


ARTICLE 1
TITLE:
LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO]

ARTICLE TEXT:
No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for the lynching and hanging of white people and cops. They encouraged others on a radio show Tuesday night to  turn the tide  and kill white people and cops to send a message about the killing of black people in America.One of the F***YoFlag organizers is called  Sunshine.  She has a radio blog show hosted from Texas called,  Sunshine s F***ing Opinion Radio Show. A snapshot of h

LABEL:
1

ARTICLE 2
TITLE:
nan

ARTICLE TEXT:
Did they post their votes for Hillary already?

LABEL:
1

ARTICLE 3
TITLE:
UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFUL” PROTESTERS…In Her Home State Of North Carolina [VIDEO]

ARTICLE TEXT:
Now, most of the demonstrators gathered last night were exer

#### Investigate missing titles

In [5]:
# Check actual missing values
print("Missing titles:", df["title"].isna().sum())
print("Missing article bodies:", df["text"].isna().sum())

# Check literal 'nan' strings
print(
    "Literal 'nan' titles:",
    df["title"].astype(str).str.strip().str.lower().eq("nan").sum()
)

# Inspect the second article
print("\nSecond article title:", repr(df.loc[1, "title"]))
print("Second article text:", repr(df.loc[1, "text"][:200]))

Missing titles: 518
Missing article bodies: 568
Literal 'nan' titles: 0

Second article title: nan
Second article text: 'Did they post their votes for Hillary already?'


In [6]:
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nEmpty titles:", df["title"].eq("").sum())
print("Empty article bodies:", df["text"].eq("").sum())

display(df.head())

Dataset loaded successfully!
Dataset shape: (63676, 3)

Missing values:
title    518
text     568
label      0
dtype: int64

Empty titles: 0
Empty article bodies: 0


,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,NaN,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last n...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [8]:
# Convert missing title and text values into empty strings
df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")

print("Dataset loaded:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nEmpty titles:", df["title"].eq("").sum())
print("Empty article bodies:", df["text"].eq("").sum())

Dataset loaded: (63676, 3)

Missing values:
title    0
text     0
label    0
dtype: int64

Empty titles: 518
Empty article bodies: 568


In [9]:
display(
    df[df["title"].eq("") | df["text"].eq("")][
        ["title", "text", "label"]
    ].head(10)
)

,title,text,label
1,,Did they post their votes for Hillary already?,1
7,HOUSE INTEL CHAIR On Trump-Russia Fake Story: ...,,1
43,,True. Hillary needs a distraction and what bet...,1
106,MARKETWATCH LEFTIST: MSM’s “Blatant” Anti Trum...,,1
162,,All eyes on Electoral delegates. The People kn...,1
185,,Cool,1
269,,A leading US senator: US Supporting War in Syr...,1
318,,Federally fund democracy by federally funding ...,1
325,Ohio State University Student Says Terrorist A...,,1
358,,Obamas blunder.,1


#### Text preprocessing function

In [10]:
def preprocess_text(text):

    if not isinstance(text, str):
        return ""

    # Decode HTML entities
    text = html.unescape(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Replace URLs with a placeholder
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " URL ",
        text
    )

    # Convert text to lowercase
    text = text.lower()

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

#### Combine title and article body

In [11]:
df["combined_text"] = (
    df["title"] + " " + df["text"]
).str.strip()

# Apply preprocessing
df["processed_text"] = df["combined_text"].apply(
    preprocess_text
)

print("Text combination completed!")

display(
    df[["title", "text", "processed_text"]].head(5)
)

Text combination completed!


,title,text,processed_text
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,law enforcement on high alert following threat...
1,,Did they post their votes for Hillary already?,did they post their votes for hillary already?
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last n...",unbelievable! obama’s attorney general says mo...
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,"bobby jindal, raised hindu, uses story of chri..."
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",satan 2: russia unvelis an image of its terrif...


#### Compare original and processed text

In [12]:
sample_index = 0

print("ORIGINAL TEXT")
print("=" * 70)
print(df.loc[sample_index, 'combined_text'][:1000])

print("\nPROCESSED TEXT")
print("=" * 70)
print(df.loc[sample_index, "processed_text"][:1000])

ORIGINAL TEXT
LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO] No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for the lynching and hanging of white people and cops. They encouraged others on a radio show Tuesday night to  turn the tide  and kill white people and cops to send a message about the killing of black people in America.One of the F***YoFlag organizers is called  Sunshine.  She has a radio blog show hosted from Texas called,  Sunshine s F***ing Opinion Radio Show. A snapshot of her #FYF911 @LOLatWhiteFear Twitter page at 9:53 p.m. shows that she was urging supporters to  Call now!! #fyf911 tonight we continue to dismantle the illusion of white Below is a SNAPSHOT Twitter Radio Call Invite   #FYF911The radio show aired at 10:00 p.m. eastern standard time.During the show, callers clearly call for  lynching  and  killing  of white

#### Validate processed data

In [14]:
print("Dataset shape:", df.shape)

print(
    "Empty processed records:",
    df["processed_text"].str.strip().eq("").sum()
)

print("\nProcessed text word-count statistics:")

df["processed_word_count"] = (
    df["processed_text"].str.split().str.len()
)

display(df["processed_word_count"].describe())

Dataset shape: (63676, 5)
Empty processed records: 0

Processed text word-count statistics:


count    63676.000000
mean       554.102849
std        610.594334
min          1.000000
25%        246.000000
50%        413.000000
75%        685.000000
max      24243.000000
Name: processed_word_count, dtype: float64

#### Save the preprocessed dataset

In [15]:
# Select the columns needed for machine learning
final_df = df[["processed_text", "label"]].copy()

# Save the preprocessed dataset to a new CSV file
final_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

print("Preprocessed dataset saved successfully!")

print("\nFile location:")
print(OUTPUT_PATH.resolve())

print("\nFinal dataset shape:", final_df.shape)

print("\nColumns:")
print(final_df.columns.tolist())

Preprocessed dataset saved successfully!

File location:
C:\Projects\fake_news_detection_extension\backend\data\WELFake_preprocessed.csv

Final dataset shape: (63676, 2)

Columns:
['processed_text', 'label']


In [16]:
# Verify the saved dataset
check_df = pd.read_csv(OUTPUT_PATH)

print("File exists:", OUTPUT_PATH.exists())
print("Verified shape:", check_df.shape)
print("Missing values:")
print(check_df.isnull().sum())

display(check_df.head())

File exists: True
Verified shape: (63676, 2)
Missing values:
processed_text    0
label             0
dtype: int64


,processed_text,label
0,law enforcement on high alert following threat...,1
1,did they post their votes for hillary already?,1
2,unbelievable! obama’s attorney general says mo...,1
3,"bobby jindal, raised hindu, uses story of chri...",0
4,satan 2: russia unvelis an image of its terrif...,1


In [17]:
check_df = pd.read_csv(OUTPUT_PATH)

print("Dataset shape:", check_df.shape)
print("Columns:", check_df.columns.tolist())

display(check_df.head())

Dataset shape: (63676, 2)
Columns: ['processed_text', 'label']


,processed_text,label
0,law enforcement on high alert following threat...,1
1,did they post their votes for hillary already?,1
2,unbelievable! obama’s attorney general says mo...,1
3,"bobby jindal, raised hindu, uses story of chri...",0
4,satan 2: russia unvelis an image of its terrif...,1
